# LinkedIn Zip Solver — Setup & Run Notebook (v2)

This notebook provisions the environment, verifies all modules, trains the CNN, and launches the LinkedIn Zip solver.

**What's new in v2:**
- DOM-first vision (reads `[data-cell-idx]` directly — pixel-perfect, no OCR)
- Edge wall detection via OpenCV (internal thick black walls)
- Optimized solver (Warnsdorff heuristic + deadlock detection + CNN prior)
- Blazing fast executor (direct cell-to-cell mouse movement)
- Trained CNN weights shipped as `zip_cnn.pth`

## 0. Configuration

Set `LOGGED_IN = True` when you are logged into LinkedIn and want to solve your **daily puzzle** (uses your saved browser profile with cookies).

Set `LOGGED_IN = False` to open a **fresh guest browser** — navigate to LinkedIn Zip, click Start Game, and the solver will read and solve the real puzzle. **No login needed, unlimited plays.**

In [ ]:
# ========================== CONFIG ==========================
LOGGED_IN = False  # True = use saved login, False = guest browser
# ============================================================

print(f'Mode: {"LOGGED IN (daily puzzle)" if LOGGED_IN else "GUEST (unlimited practice)"}')

## 1. Environment Check

In [ ]:
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root))

print('Python:', sys.executable)
print('Version:', sys.version)
print('Repo root:', repo_root)
print()

# Check installed packages.
packages = ['playwright', 'torch', 'torchvision', 'cv2', 'numpy', 'PIL']
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f'  {pkg:15s} {ver}')
    except ImportError:
        print(f'  {pkg:15s} MISSING')

## 2. Install Missing Dependencies

If any packages are missing, this cell installs them automatically.

In [ ]:
import subprocess

requirements = [
    'playwright>=1.40.0',
    'torch>=2.1.0',
    'torchvision>=0.16.0',
    'opencv-python>=4.8.0',
    'numpy>=1.24.0',
    'pillow>=10.0.0',
]

missing = []
for req in requirements:
    pkg = req.split('>=')[0]
    try:
        __import__(pkg if pkg != 'opencv-python' else 'cv2')
    except ImportError:
        missing.append(req)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
    print('Done. Please restart the kernel and re-run this cell.')
else:
    print('All dependencies are installed.')

# Also ensure Playwright's Chromium is installed.
try:
    subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])
except Exception as e:
    print('Playwright install note:', e)

## 3. Import All Modules

Verify that every module loads without errors.

In [ ]:
import main
import model
import solver
import vision
import executor
import train

print('All modules imported successfully:')
print('  vision, solver, model, executor, train, main')

## 4. CNN Model Smoke Test

Load the trained CNN (if weights exist) and run a single inference.

In [ ]:
from model import load_model, NUM_CHANNELS, HAS_TORCH
import torch

print(f'PyTorch available: {HAS_TORCH}')

if HAS_TORCH:
    model_obj = load_model()
    if model_obj is not None:
        params = sum(p.numel() for p in model_obj.parameters())
        print(f'Model loaded: {params:,} parameters')
        
        # Dummy inference.
        dummy = torch.zeros(1, NUM_CHANNELS, 8, 8)
        policy, value = model_obj.predict(dummy)
        print(f'Policy probabilities: {policy.tolist()}')
        print(f'Value score: {value:.4f}')
    else:
        print('Model not loaded (torch available but load failed).')
else:
    print('torch not installed — solver will use plain DFS.')

## 5. Solver Test — 5 Synthetic Boards

Run the solver on 5 randomly generated boards and verify each solution is correct.

In [ ]:
import time
import random
import numpy as np
from vision import generate_synthetic_state
from solver import ZipSolver, SolverConfig
from executor import validate_path, build_drag_path, ExecutorConfig

config = SolverConfig(
    max_iterations=2_000_000,
    use_cnn_prior=True,
    use_heuristic_prior=True,
    use_deadlock_check=True,
    verbose=False,
)

all_passed = True
for seed in range(5):
    state, true_path = generate_synthetic_state(rows=6, cols=6, seed=seed)
    solver_inst = ZipSolver(state, model=model_obj, config=config)
    t0 = time.time()
    path = solver_inst.solve()
    elapsed = time.time() - t0
    
    if path is None:
        print(f'Seed {seed}: FAIL — no solution')
        all_passed = False
        continue
    
    ok = (validate_path(path) and
          path[0] == state.start and
          path[-1] == state.goal and
          len(set(path)) == len(path) and
          len(path) == len(true_path))
    
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_passed = False
    print(f'Seed {seed}: {status} — {len(path)} cells, '
          f'{solver_inst.nodes_explored} nodes, {elapsed:.3f}s')

print(f'\nSolver test: {"ALL PASSED" if all_passed else "SOME FAILED"}')

## 6. DOM Parsing Test

Test the DOM-based vision parser on a synthetic DOM state (simulates what `page.evaluate()` returns from LinkedIn).

In [ ]:
from vision import build_state_from_dom, CellType

# Build a synthetic 4x4 DOM with walls.
rows, cols = 4, 4
numbered = {0: 1, 3: 2, 15: 3}  # start, mid, goal
wall_indices = {5, 7, 12}
active_bg = 'rgb(255, 255, 255)'
wall_bg = 'rgb(230, 220, 200)'

cells_data = []
for r in range(rows):
    for c in range(cols):
        idx = r * cols + c
        if idx in wall_indices:
            bg, num = wall_bg, None
        elif idx in numbered:
            bg, num = active_bg, numbered[idx]
        else:
            bg, num = active_bg, None
        cells_data.append({
            'idx': idx, 'number': num,
            'ariaLabel': str(num) if num else None,
            'bg': bg, 'x': c * 50, 'y': r * 50, 'w': 50, 'h': 50,
        })

dom_state = {
    'count': rows * cols,
    'cells': cells_data,
    'gridRect': {'x': 0, 'y': 0, 'w': cols * 50, 'h': rows * 50},
}

state = build_state_from_dom(dom_state)
print(f'Parsed grid ({state.shape[0]}x{state.shape[1]}):')
for r in range(rows):
    row_str = '  '
    for c in range(cols):
        ct = state.grid[r, c]
        if ct == int(CellType.WALL):
            row_str += ' W '
        elif ct == int(CellType.START):
            row_str += ' S '
        elif ct == int(CellType.GOAL):
            row_str += ' G '
        elif ct == int(CellType.NUMBERED):
            for cell in state.cells:
                if cell.row == r and cell.col == c:
                    row_str += f' {cell.number} '
                    break
        else:
            row_str += ' . '
    print(row_str)

print(f'Start: {state.start}, Goal: {state.goal}')
print(f'Waypoints: {state.numbers.tolist()}')
print(f'Walls detected: {sum(1 for r in range(rows) for c in range(cols) if state.grid[r,c] == int(CellType.WALL))}')

## 7. Train the CNN (Optional)

If `zip_cnn.pth` doesn't exist or you want to retrain, run this cell. It generates 200 synthetic games and trains for 15 epochs (~1-2 minutes on CPU).

In [ ]:
from pathlib import Path
weights_path = Path('zip_cnn.pth')
if weights_path.exists():
    print(f'Weights already exist: {weights_path} ({weights_path.stat().st_size / 1024:.1f} KB)')
    print('To retrain, delete the file and re-run this cell.')
else:
    print('Training CNN...')
    import subprocess
    result = subprocess.run(
        [sys.executable, 'train.py', '--num-games', '200', '--epochs', '15'],
        cwd=str(repo_root), text=True, capture_output=True
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    if weights_path.exists():
        print(f'Training complete: {weights_path.stat().st_size / 1024:.1f} KB')

## 8. Run the Solver

Both modes open a **real Chrome browser** and solve the **real LinkedIn Zip puzzle**.

- **`LOGGED_IN = True`** → Uses your saved browser profile (with LinkedIn cookies). Solves your daily puzzle.
- **`LOGGED_IN = False`** → Opens a fresh guest browser (not logged in). Navigate to the puzzle, click **Start Game**, and the solver will read and solve it. **Unlimited plays.**

**Tip:** When the browser opens, wait for the puzzle board to fully load before the solver begins capturing.

In [ ]:
import subprocess

if LOGGED_IN:
    print('🌐 LOGGED IN — Solving your daily LinkedIn Zip puzzle...')
    print('Using your saved browser profile.')
else:
    print('🧪 GUEST MODE — Opening a fresh browser (not logged in)...')
    print('Navigate to the puzzle and click Start Game. Unlimited plays!')

print()

cmd = [sys.executable, 'main.py', '--solve-only', '--skip-setup']
if not LOGGED_IN:
    cmd.append('--guest')

result = subprocess.run(
    cmd,
    cwd=str(repo_root),
    text=True,
    capture_output=False
)
print(f'\nExit code: {result.returncode}')